In [1]:
import os
import numpy as np
import dask
import dask.array as da
from dask.diagnostics import ProgressBar
from pathlib import Path
import rasterio
from torch.utils.data import Dataset

import glob
import torch
from torch.utils.data import DataLoader

try:
    import cudf
    import cupy as cp
    BACKEND = 'gpu'
    print("✓ GPU backend: CuDF + CuPy available")
except ImportError:
    BACKEND = 'cpu'
    print("✓ CPU backend: Dask + NumPy")

from config import CFG

✓ GPU backend: CuDF + CuPy available


In [2]:
CFG.USEATTENTION

False

In [3]:
@dask.delayed
def readtif(filepath: str, shape: tuple) -> np.ndarray:
    """Delayed single GeoTIFF read.

    Ensures the returned array has exactly the requested (H, W) shape by
    padding with NaNs or cropping from the bottom/right if necessary.
    This avoids Dask broadcasting errors when some rasters are off by
    one pixel compared to the reference size.
    """
    H, W = shape

    # If file missing, return an all-NaN array with the expected shape
    if not os.path.exists(filepath):
        return np.full((H, W), np.nan, dtype=np.float32)

    with rasterio.open(filepath) as src:
        arr = src.read(1).astype(np.float32)
        nodata = src.nodata

    if nodata is not None:
        arr[arr == nodata] = np.nan

    h, w = arr.shape
    if (h, w) == (H, W):
        return arr

    # Create target canvas and copy overlap; handles both smaller and larger inputs
    out = np.full((H, W), np.nan, dtype=np.float32)
    hh = min(h, H)
    ww = min(w, W)
    out[:hh, :ww] = arr[:hh, :ww]
    return out

def getrastershape(dataroot: str) -> tuple:
    """Infer (H, W) from the first available GeoTIFF."""
    for var in CFG.BIWEEKLYVARS + [CFG.TARGETVAR]:
        vardir = Path(dataroot) / var
        files = sorted(vardir.glob('*.tif'))
        if files:
            with rasterio.open(files[0]) as src:
                return src.height, src.width, src.crs, src.transform
    raise RuntimeError("No GeoTIFF files found. Check DATAROOT.")

def builddatastack(dataroot=CFG.DATAROOT, years=None, verbose=True):
    """
    Build lazy (T, C, H, W) Dask arrays for inputs and targets.
    Returns: Xdask, Ydask, timestamps, rastermeta
    """
    if years is None:
        # Default: build a single continuous stack over train/val/test years
        years = CFG.TRAINYEARS + CFG.VALYEARS + CFG.TESTYEARS

    # Infer reference raster shape and georeferencing from any available file
    H, W, crs, transform = getrastershape(dataroot)
    rastermeta = {'height': H, 'width': W, 'crs': crs, 'transform': transform}
    if verbose:
        print(f"  Raster: {H}×{W} px | CRS: {crs}")

    def biweekly_path(var: str, year: int, doty: int) -> str:
        """Return a valid path for a biweekly variable, handling 0-padded/non‑padded doty.

        Some datasets name files as var_YYYY_DDD.tif (DDD=001,017,...) while others
        omit zero padding (e.g. var_YYYY_1.tif). We try the padded name first for
        consistency, then fall back to the non-padded variant.
        """
        base = Path(dataroot) / var
        cand1 = base / f"{var}_{year}_{doty:03d}.tif"  # zero‑padded doty
        if cand1.exists():
            return str(cand1)
        cand2 = base / f"{var}_{year}_{doty}.tif"       # no padding
        return str(cand2)

    def target_path(year: int, doty: int) -> str:
        """Return a valid path for the target raster, handling 0-padded/non‑padded doty.

        Same padded/non-padded logic as for biweekly inputs, but for the watertable
        target variable.
        """
        base = Path(dataroot) / CFG.TARGETVAR
        cand1 = base / f"watertable_{year}_{doty:03d}.tif"
        if cand1.exists():
            return str(cand1)
        cand2 = base / f"watertable_{year}_{doty}.tif"
        return str(cand2)

    Xsteps, Ysteps, timestamps = [], [], []

    # Iterate over all requested years and biweekly day-of-year grid
    for year in years:
        for doty in CFG.BIWEEKLYDOTY:
            channels = []

            # 1) Biweekly variables (time-varying covariates)
            for var in CFG.BIWEEKLYVARS:
                fp = biweekly_path(var, year, doty)
                # Each delayed read produces a (H, W) array for that (var, year, doty)
                arr = da.from_delayed(
                    readtif(fp, (H, W)), shape=(H, W), dtype=np.float32
                )
                channels.append(arr[np.newaxis])  # add channel axis → (1, H, W)

            # 2) Yearly variables (static within a year, e.g. landuse)
            for var in CFG.YEARLYVARS:
                fp = str(Path(dataroot) / var / f"{var}_{year}.tif")
                arr = da.from_delayed(
                    readtif(fp, (H, W)), shape=(H, W), dtype=np.float32
                )
                channels.append(arr[np.newaxis])

            # Stack all channels for this (year, doty) into (C, H, W)
            stepx = da.concatenate(channels, axis=0)
            # Add time-axis, so each entry is (1, C, H, W)
            Xsteps.append(stepx[np.newaxis])

            # Target (watertable) for the same (year, doty)
            fpy = target_path(year, doty)
            arry = da.from_delayed(
                readtif(fpy, (H, W)), shape=(H, W), dtype=np.float32
            )
            Ysteps.append(arry[np.newaxis, np.newaxis])  # (1, 1, H, W)

            # Record mapping from time index → (year, doty)
            timestamps.append((year, doty))

    # Concatenate along time dimension → (T, C, H, W) and (T, 1, H, W)
    Xdask = da.concatenate(Xsteps, axis=0)
    Ydask = da.concatenate(Ysteps, axis=0)

    if verbose:
        print(f"  Xdask: {Xdask.shape}  |  Ydask: {Ydask.shape}")
        print(f"  Timesteps: {len(timestamps)} ({years[0]}→{years[-1]})")

    return Xdask, Ydask, timestamps, rastermeta

def computenormstats(Xdask, savepath=CFG.NORMSTATS):
    """Compute per-channel Z-score statistics lazily via Dask. Use TRAINING split only."""
    C = Xdask.shape[1]
    print(f"  Computing normalisation stats for {C} channels...")
    meantasks = [da.nanmean(Xdask[:, c]) for c in range(C)]
    stdtasks  = [da.nanstd(Xdask[:, c])  for c in range(C)]
    
    with ProgressBar():
        results = dask.compute(meantasks, stdtasks, scheduler='threads')
    
    means = np.array(results[0], dtype=np.float32)
    stds  = np.array(results[1], dtype=np.float32)
    stds  = np.where(stds < 1e-8, 1.0, stds)
    
    np.savez(savepath, means=means, stds=stds)
    print(f"  Saved norm stats → {savepath}")
    return means, stds

def loadnormstats(path=CFG.NORMSTATS):
    d = np.load(path)
    return d['means'], d['stds']

def applynorm(Xdask, means, stds):
    m = means[np.newaxis, :, np.newaxis, np.newaxis]
    s = stds [np.newaxis, :, np.newaxis, np.newaxis]
    return (Xdask - m) / s

def computetonumpy(daskdict: dict, scheduler='threads') -> dict:
    names  = list(daskdict.keys())
    arrays = list(daskdict.values())
    print(f"  Computing {len(arrays)} arrays to NumPy (scheduler='{scheduler}')...")
    with ProgressBar():
        results = dask.compute(*arrays, scheduler=scheduler)
    return {n: r for n, r in zip(names, results)}

In [4]:
class GroundwaterDataset(Dataset):
    """Each sample: xseq (SEQLEN, C, PATCH, PATCH), yseq (PREDHORIZON, 1, PATCH, PATCH).

    Turns the full (T, C, H, W) arrays into overlapping spatio-temporal
    patches with a lookback window of SEQLEN steps and a forecast window
    of PREDHORIZON steps.
    """

    def __init__(self, X, Y, seqlen=CFG.SEQLEN, predhorizon=CFG.PREDHORIZON,
                 patchsize=CFG.PATCHSIZE, stride=CFG.PATCHSTRIDE,
                 augment=False, fillnan=0.0):
        # Store references to the full normalised arrays
        self.X, self.Y = X, Y
        self.seqlen, self.predhor = seqlen, predhorizon
        self.ps, self.stride = patchsize, stride
        self.augment, self.fillnan = augment, fillnan

        # Cache global dims for fast index computation
        T, C, H, W = X.shape
        self.T, self.H, self.W = T, H, W

        # Pre-compute all valid (t, i, j) triplets once
        self.indices = self._build_indices()
        print(f"  Dataset ready — {len(self.indices):,} samples (T={T}, H={H}, W={W}, "
              f"patch={patchsize}, stride={stride})")

    def _build_indices(self):
        """Return list of (t, i, j) where the target patch is not fully missing."""
        ps, st = self.ps, self.stride
        # Last starting time index that still fits seqlen + predhor steps
        maxt = self.T - self.seqlen - self.predhor + 1
        indices = []
        for t in range(maxt):
            for i in range(0, self.H - ps + 1, st):
                for j in range(0, self.W - ps + 1, st):
                    # Target patch at the first forecast step
                    ypatch = self.Y[t + self.seqlen, 0, i:i+ps, j:j+ps]
                    # Skip patches where the target is completely NaN
                    if not np.isnan(ypatch).all():
                        indices.append((t, i, j))
        return indices

    def __len__(self):
        # Number of valid spatio-temporal samples
        return len(self.indices)

    def __getitem__(self, idx):
        # Look up the precomputed (t, i, j) triple
        t, i, j = self.indices[idx]
        ps = self.ps

        # Extract input and target sequences as NumPy views
        x = self.X[t : t + self.seqlen, :, i:i+ps, j:j+ps].copy()
        y = self.Y[t + self.seqlen : t + self.seqlen + self.predhor, :, i:i+ps, j:j+ps].copy()

        # Replace NaNs with a fixed fill value for model input
        np.nan_to_num(x, copy=False, nan=self.fillnan)
        np.nan_to_num(y, copy=False, nan=self.fillnan)

        # Convert to PyTorch tensors
        x = torch.from_numpy(x).float()
        y = torch.from_numpy(y).float()

        # Optional simple geometric augmentation (random flips)
        if self.augment:
            if torch.rand(1).item() > 0.5:
                x, y = torch.flip(x, dims=[-1]), torch.flip(y, dims=[-1])  # horizontal flip
            if torch.rand(1).item() > 0.5:
                x, y = torch.flip(x, dims=[-2]), torch.flip(y, dims=[-2])  # vertical flip

        return x, y

In [5]:
%%writefile model.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from config import CFG

class ConvBlock(nn.Module):
    def __init__(self, inch, outch, kernel=3):
        super().__init__()
        pad = kernel // 2
        self.net = nn.Sequential(
            nn.Conv2d(inch, outch, kernel, padding=pad, bias=False),
            nn.BatchNorm2d(outch), nn.ReLU(inplace=True),
            nn.Conv2d(outch, outch, kernel, padding=pad, bias=False),
            nn.BatchNorm2d(outch), nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)

class UpBlock(nn.Module):
    def __init__(self, inch, skipch, outch):
        super().__init__()
        self.up = nn.ConvTranspose2d(inch, outch, kernel_size=2, stride=2)
        self.conv = ConvBlock(outch + skipch, outch)

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))

class CNNEncoder(nn.Module):
    def __init__(self, inchannels, filters):
        super().__init__()
        f1, f2, f3 = filters
        self.block1 = ConvBlock(inchannels, f1)
        self.pool1 = nn.MaxPool2d(2)
        self.block2 = ConvBlock(f1, f2)
        self.pool2 = nn.MaxPool2d(2)
        self.block3 = ConvBlock(f2, f3)
        self.pool3 = nn.MaxPool2d(2)
        self.gap = nn.AdaptiveAvgPool2d(1)

    def forward(self, x):
        s1 = self.block1(x)
        s2 = self.block2(self.pool1(s1))
        s3 = self.block3(self.pool2(s2))
        bottleneck = self.pool3(s3)
        feats = self.gap(bottleneck).flatten(1)
        return feats, [s1, s2, s3, bottleneck]

class BiLSTMEncoder(nn.Module):
    def __init__(self, inputdim, hiddendim, nlayers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(
            inputdim, hiddendim, nlayers,
            batch_first=True,
            dropout=dropout if nlayers > 1 else 0.0,
            bidirectional=True
        )
        self.outdim = hiddendim * 2

    def forward(self, x):
        out, _ = self.lstm(x)
        return out

class TemporalAttention(nn.Module):
    def __init__(self, hiddendim, nheads=4, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(hiddendim, nheads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(hiddendim)

    def forward(self, x):
        out, _ = self.attn(x, x, x)
        out = self.norm(out + x)
        return out[:, -1]

class SpatialDecoder(nn.Module):
    def __init__(self, contextdim, filters, predhorizon, patchsize):
        super().__init__()
        f1, f2, f3 = filters
        h8 = patchsize // 8
        self.h8, self.f3 = h8, f3
        self.fcproj = nn.Sequential(
            nn.Linear(contextdim, f3 * h8 * h8),
            nn.ReLU(inplace=True)
        )
        self.fuse = ConvBlock(f3 * 2, f3)
        self.up1 = UpBlock(f3, f3, f2)
        self.up2 = UpBlock(f2, f2, f1)
        self.up3 = UpBlock(f1, f1, 32)
        self.outconv = nn.Conv2d(32, predhorizon, kernel_size=1)

    def forward(self, context, skips):
        s1, s2, s3, bottleneck = skips
        B = context.size(0)
        proj = self.fcproj(context).view(B, self.f3, self.h8, self.h8)
        x = self.fuse(torch.cat([proj, bottleneck], dim=1))
        x = self.up1(x, s3)
        x = self.up2(x, s2)
        x = self.up3(x, s1)
        out = self.outconv(x)
        return out.unsqueeze(2)

class CNNLSTMForecaster(nn.Module):
    def __init__(self, config=CFG):
        super().__init__()
        self.config = config
        self.encoder = CNNEncoder(config.NCHANNELS, config.CNNCHANNELS)
        encdim = config.CNNCHANNELS[-1]
        self.temporal = BiLSTMEncoder(encdim, config.LSTMHIDDEN, config.LSTMLAYERS, config.DROPOUT)
        lstmdim = config.LSTMHIDDEN * 2
        self.attn = (TemporalAttention(lstmdim, config.ATTNHEADS, config.DROPOUT)
                     if config.USEATTENTION else None)
        self.decoder = SpatialDecoder(lstmdim, config.CNNCHANNELS, config.PREDHORIZON, config.PATCHSIZE)
        self.drop = nn.Dropout(config.DROPOUT)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            elif isinstance(m, (nn.BatchNorm2d, nn.LayerNorm)):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)

    def forward(self, x):
        B, T, C, H, W = x.shape
        xflat = x.view(B * T, C, H, W)
        featsflat, skipsflat = self.encoder(xflat)
        Fdim = featsflat.size(1)
        temporalin = featsflat.view(B, T, Fdim)
        temporalin = self.drop(temporalin)
        lstmout = self.temporal(temporalin)
        context = self.attn(lstmout) if self.attn else lstmout[:, -1]
        context = self.drop(context)
        lastskips = []
        for skip in skipsflat:
            _, ch, sh, sw = skip.shape
            lastskips.append(skip.view(B, T, ch, sh, sw)[:, -1])
        return self.decoder(context, lastskips)

    def count_params(self):
        n = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"  Trainable parameters: {n:,}")
        return n

Overwriting model.py


In [6]:
%%writefile utils.py
import os
import numpy as np
import torch
import logging

logging.basicConfig(format='%(asctime)s | %(levelname)s | %(message)s', level=logging.INFO)
log = logging.getLogger(__name__)

class EarlyStopping:
    def __init__(self, patience=15, delta=1e-5, checkpoint_path='checkpoints/bestmodel.pt'):
        self.patience = patience
        self.delta = delta
        self.path = checkpoint_path
        self.counter = 0
        self.best_loss = float('inf')

    def __call__(self, val_loss, model) -> bool:
        if val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.counter = 0
            save_checkpoint(model, self.path)
            log.info(f"  ✓ New best val loss: {val_loss:.5f} — checkpoint saved.")
            return False
        else:
            self.counter += 1
            log.info(f"  EarlyStopping counter: {self.counter}/{self.patience}")
            return self.counter >= self.patience

def save_checkpoint(model, path, optimizer=None, epoch=None):
    os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
    m = model.module if isinstance(model, torch.nn.DataParallel) else model
    state = {'model_state': m.state_dict(), 'epoch': epoch}
    if optimizer:
        state['optim_state'] = optimizer.state_dict()
    torch.save(state, path)

def load_checkpoint(model, path, optimizer=None, device='cpu'):
    state = torch.load(path, map_location=device)
    m = model.module if isinstance(model, torch.nn.DataParallel) else model
    m.load_state_dict(state['model_state'])
    if optimizer and 'optim_state' in state:
        optimizer.load_state_dict(state['optim_state'])
    epoch = state.get('epoch', 0)
    log.info(f"  Loaded checkpoint from '{path}' (epoch {epoch})")
    return epoch

def compute_metrics(preds: np.ndarray, targets: np.ndarray, eps=1e-8) -> dict:
    p = preds.flatten().astype(np.float64)
    t = targets.flatten().astype(np.float64)
    mask = ~(np.isnan(p) | np.isnan(t))
    p, t = p[mask], t[mask]
    rmse = float(np.sqrt(np.mean((p - t) ** 2)))
    mae = float(np.mean(np.abs(p - t)))
    ss_res = np.sum((t - p) ** 2)
    ss_tot = np.sum((t - t.mean()) ** 2) + eps
    r2 = float(1.0 - ss_res / ss_tot)
    nse = r2
    pbias = float(100 * (p.sum() - t.sum()) / (t.sum() + eps))
    return {'rmse': rmse, 'mae': mae, 'r2': r2, 'nse': nse, 'pbias': pbias}

def print_metrics(metrics: dict, prefix=''):
    print(f"{prefix}RMSE={metrics['rmse']:.4f} | MAE={metrics['mae']:.4f} | "
          f"NSE={metrics['nse']:.4f} | R²={metrics['r2']:.4f} | PBIAS={metrics['pbias']:.2f}%")

Overwriting utils.py


In [7]:
%%writefile train.py
import time
import torch
import torch.nn as nn
import numpy as np
from torch.cuda.amp import GradScaler, autocast
from torch.optim.lr_scheduler import CosineAnnealingLR, StepLR, ReduceLROnPlateau
from torch.utils.tensorboard import SummaryWriter

from config import CFG
from utils import EarlyStopping, compute_metrics, print_metrics, log

def train_one_epoch(model, loader, optimizer, criterion, scaler, device, grad_clip):
    model.train()
    total_loss, n = 0.0, 0
    for batch_idx, (x, y) in enumerate(loader):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        # turn off autocast → full FP32 training
        # (you can remove autocast import later if you like)
        with autocast(enabled=False):
            pred = model(x)
            loss = criterion(pred, y)

        # NaN/inf check
        if not torch.isfinite(loss):
            print(f"[NaN loss] batch {batch_idx}")
            print("  x finite:", torch.isfinite(x).all().item())
            print("  y finite:", torch.isfinite(y).all().item())
            print("  pred finite:", torch.isfinite(pred).all().item())
            print("  pred range:", float(torch.nanmin(pred)), float(torch.nanmax(pred)))
            raise RuntimeError("Loss became non-finite")

        # no AMP scaling
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()

        total_loss += float(loss.item())
        n += 1

    return total_loss / max(n, 1)

@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss, n = 0.0, 0
    all_preds, all_targets = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        pred = model(x)
        loss = criterion(pred, y)
        total_loss += loss.item()
        n += 1
        all_preds.append(pred.cpu().float().numpy())
        all_targets.append(y.cpu().float().numpy())
    preds = np.concatenate(all_preds, axis=0)
    targets = np.concatenate(all_targets, axis=0)
    metrics = compute_metrics(preds, targets)
    return total_loss / max(n, 1), metrics

def build_scheduler(optimizer, cfg):
    if cfg.SCHEDULER == 'cosine':
        return CosineAnnealingLR(optimizer, T_max=cfg.EPOCHS, eta_min=1e-7)
    elif cfg.SCHEDULER == 'step':
        return StepLR(optimizer, step_size=30, gamma=0.5)
    elif cfg.SCHEDULER == 'plateau':
        return ReduceLROnPlateau(optimizer, patience=8, factor=0.5, verbose=True, min_lr=1e-7)
    return None

def train(model, train_loader, val_loader, cfg=CFG):
    device = cfg.DEVICE
    model = model.to(device)
    if torch.cuda.device_count() > 1:
        log.info(f"Using {torch.cuda.device_count()} GPUs via DataParallel")
        model = nn.DataParallel(model)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHTDECAY)
    criterion = nn.MSELoss()
    scheduler = build_scheduler(optimizer, cfg)
    #scaler = GradScaler(enabled=(str(device) == 'cuda'))
    scaler = GradScaler(enabled=False)
    stopper = EarlyStopping(patience=cfg.PATIENCE, checkpoint_path=f"{cfg.CHECKPOINTDIR}/bestmodel.pt")
    writer = SummaryWriter(log_dir=cfg.LOGDIR)

    log.info(f"\n{'═'*65}\n  Device: {device}  |  Epochs: {cfg.EPOCHS}  |  Patience: {cfg.PATIENCE}\n  LR: {cfg.LR}  |  Scheduler: {cfg.SCHEDULER}\n{'═'*65}\n")

    history = {k: [] for k in ['train_loss', 'val_loss', 'rmse', 'mae', 'nse', 'r2']}

    for epoch in range(1, cfg.EPOCHS + 1):
        t0 = time.time()
        tr_loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler, device, cfg.GRADCLIP)
        val_loss, met = validate(model, val_loader, criterion, device)
        elapsed = time.time() - t0
        if scheduler:
            if cfg.SCHEDULER == 'plateau':
                scheduler.step(val_loss)
            else:
                scheduler.step()

        writer.add_scalars('Loss', {'train': tr_loss, 'val': val_loss}, epoch)
        writer.add_scalar('Metrics/RMSE', met['rmse'], epoch)
        writer.add_scalar('Metrics/NSE', met['nse'], epoch)
        writer.add_scalar('LR', optimizer.param_groups[0]['lr'], epoch)

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(val_loss)
        for k in ['rmse', 'mae', 'nse', 'r2']:
            history[k].append(met[k])

        log.info(f"Epoch {epoch:3d}/{cfg.EPOCHS} | Train {tr_loss:.4f} | Val {val_loss:.4f} | NSE {met['nse']:.4f} | RMSE {met['rmse']:.4f} | {elapsed:.1f}s")

        if stopper(val_loss, model):
            log.info("\n⏹ Early stopping at epoch %d.", epoch)
            break

    writer.close()
    np.save(f"{cfg.RESULTSDIR}/history.npy", history)
    log.info("Training complete.")
    return history

Overwriting train.py


In [8]:
%%writefile forecast.py
import os
import numpy as np
import torch
import rasterio
from config import CFG
from utils import compute_metrics, print_metrics, log

@torch.no_grad()
def forecast_full_raster(model, X_window, cfg=CFG, device=CFG.DEVICE, batch_size=16) -> np.ndarray:
    """Predict water-table depth for an entire raster via sliding-window inference. X_window: (SEQLEN, C, H, W). Returns: (PREDHORIZON, 1, H, W)."""
    model.eval()
    _, _, H, W = X_window.shape
    ps, st = cfg.PATCHSIZE, cfg.PATCHSTRIDE
    patches, positions = [], []
    for i in range(0, H - ps + 1, st):
        for j in range(0, W - ps + 1, st):
            patches.append(X_window[:, :, i:i+ps, j:j+ps])
            positions.append((i, j))
    preds = []
    for start in range(0, len(patches), batch_size):
        batch = np.stack(patches[start:start + batch_size])
        xt = torch.from_numpy(batch).float().to(device)
        out = model(xt)
        preds.extend(out.cpu().numpy())
    P = preds[0].shape[0]
    accum = np.zeros((P, 1, H, W), dtype=np.float32)
    count = np.zeros((P, 1, H, W), dtype=np.float32)
    for pred, (r, c) in zip(preds, positions):
        accum[:, :, r:r+ps, c:c+ps] += pred
        count[:, :, r:r+ps, c:c+ps] += 1.0
    return accum / np.maximum(count, 1.0)

def season_indices(timestamps, season_name):
    s = CFG.SEASONS[season_name]
    start, end, wraps = s['start'], s['end'], s['wraps']
    idx = []
    for i, (_, doty) in enumerate(timestamps):
        in_season = (doty >= start or doty <= end) if wraps else (start <= doty <= end)
        if in_season:
            idx.append(i)
    return idx

def seasonal_forecast(model, X_norm, timestamps, cfg=CFG, device=CFG.DEVICE, y_mean=0.0, y_std=1.0) -> dict:
    results = {}
    for name in cfg.SEASONS:
        idx = season_indices(timestamps, name)
        log.info(f"\n── Season: {name}  ({len(idx)} timesteps) ──")
        if len(idx) < cfg.SEQLEN:
            log.warning("  Skipping %s: insufficient lookback.", name)
            continue
        last_t = idx[-1]
        if last_t < cfg.SEQLEN:
            log.warning("  Skipping %s: not enough history.", name)
            continue
        window = X_norm[last_t - cfg.SEQLEN : last_t]
        pred = forecast_full_raster(model, window, cfg, device)
        pred_dn = pred * y_std + y_mean
        log.info(f"  Output: {pred_dn.shape} | WT depth mean={pred_dn.mean():.3f} m  std={pred_dn.std():.3f} m")
        results[name] = pred_dn
    return results

def longterm_forecast(model, X_norm, timestamps, cfg=CFG, device=CFG.DEVICE, y_mean=0.0, y_std=1.0, n_years=CFG.FORECASTYEARS) -> np.ndarray:
    n_steps_total = n_years * len(cfg.BIWEEKLYDOTY)
    log.info(f"\n── Long-term forecast: {n_years} yr ({n_steps_total} steps) ──")
    T_total = X_norm.shape[0]
    n_doty = len(cfg.BIWEEKLYDOTY)
    climatemean = np.zeros_like(X_norm[:n_doty])
    for di, doty in enumerate(cfg.BIWEEKLYDOTY):
        didx = [i for i, (_, d) in enumerate(timestamps) if d == doty and i < T_total]
        if didx:
            climatemean[di] = np.nanmean(X_norm[didx], axis=0)
        else:
            climatemean[di] = X_norm[max(0, T_total - n_doty + di)]
    window = X_norm[T_total - cfg.SEQLEN : T_total].copy()
    all_preds = []
    steps_done = 0
    doty_list = cfg.BIWEEKLYDOTY
    try:
        start_di = doty_list.index(timestamps[-1][1])
    except ValueError:
        start_di = 0
    while steps_done < n_steps_total:
        pred = forecast_full_raster(model, window, cfg, device)
        take = min(cfg.PREDHORIZON, n_steps_total - steps_done)
        all_preds.append(pred[:take])
        steps_done += take
        if steps_done < n_steps_total:
            new_frames = []
            for k in range(cfg.PREDHORIZON):
                di = (start_di + steps_done + k) % len(doty_list)
                frame = climatemean[di].copy()
                new_frames.append(frame)
            new_arr = np.stack(new_frames)
            window = np.concatenate([window[cfg.PREDHORIZON:], new_arr], axis=0)
        log.info(f"  Progress: {min(steps_done, n_steps_total)}/{n_steps_total}")
    all_preds = np.concatenate(all_preds, axis=0)[:n_steps_total]
    return all_preds * y_std + y_mean

def save_geotiff(arr: np.ndarray, outpath: str, reference_tif: str, nodata=CFG.NODATAVAL):
    if arr.ndim == 2:
        arr = arr[np.newaxis]
    P, H, W = arr.shape
    with rasterio.open(reference_tif) as ref:
        profile = ref.profile.copy()
    profile.update(count=P, dtype='float32', nodata=nodata)
    os.makedirs(os.path.dirname(outpath) or '.', exist_ok=True)
    with rasterio.open(outpath, 'w', **profile) as dst:
        for b in range(P):
            band = arr[b].astype(np.float32)
            band[np.isnan(band)] = nodata
            dst.write(band, b + 1)
    log.info(f"  Saved GeoTIFF → {outpath}  ({P} bands)")

Overwriting forecast.py


# Demo

In [9]:
import logging
logging.basicConfig(format='%(asctime)s | %(levelname)s | %(message)s', level=logging.INFO)
log = logging.getLogger(__name__)

CFG.makedirs()
log.info("Device  : %s", CFG.DEVICE)
log.info("Workers : %s", CFG.NWORKERS)
torch.manual_seed(42)
np.random.seed(42)

2026-03-10 16:22:54,141 | INFO | Device  : cuda
2026-03-10 16:22:54,142 | INFO | Workers : 8


✓ Output directories ready.


In [10]:
Xtrd, Ytrd, tr_ts, meta = builddatastack(years=CFG.TRAINYEARS)
Xvld, Yvld, vl_ts, _ = builddatastack(years=CFG.VALYEARS)
Xtsd, Ytsd, ts_ts, _ = builddatastack(years=CFG.TESTYEARS)

  Raster: 670×480 px | CRS: EPSG:9680
  Xdask: (368, 12, 670, 480)  |  Ydask: (368, 1, 670, 480)
  Timesteps: 368 (2000→2015)
  Raster: 670×480 px | CRS: EPSG:9680
  Xdask: (46, 12, 670, 480)  |  Ydask: (46, 1, 670, 480)
  Timesteps: 46 (2016→2017)
  Raster: 670×480 px | CRS: EPSG:9680
  Xdask: (69, 12, 670, 480)  |  Ydask: (69, 1, 670, 480)
  Timesteps: 69 (2018→2020)


In [11]:
# Calculate mean and std for target variable.

if os.path.exists(CFG.NORMSTATS):
        Xmeans, Xstds = loadnormstats()
        log.info("  Loaded existing norm stats.")
else:
    Xmeans, Xstds = computenormstats(Xtrd)

2026-03-10 16:23:00,839 | INFO |   Loaded existing norm stats.


In [12]:
# Calculate mean and std for target variable.

with ProgressBar():
    y_mean, y_std = da.nanmean(Ytrd).compute(), da.nanstd(Ytrd).compute()
y_mean, y_std = float(y_mean), float(y_std)
log.info("  Target  mean=%.4f  std=%.4f", y_mean, y_std)

[########################################] | 100% Completed | 6.75 ss
[########################################] | 100% Completed | 7.01 ss


2026-03-10 16:23:15,525 | INFO |   Target  mean=4.0217  std=2.0764


In [13]:
# Normalize values across datasets.

Xtrn = applynorm(Xtrd, Xmeans, Xstds)
Xvln = applynorm(Xvld, Xmeans, Xstds)
Xtsn = applynorm(Xtsd, Xmeans, Xstds)
Ytrn = (Ytrd - y_mean) / y_std
Yvln = (Yvld - y_mean) / y_std
Ytsn = (Ytsd - y_mean) / y_std

In [14]:
# Conversion from dask datasets to Numpy matrices.

arrays = computetonumpy({'Xtr': Xtrn, 'Ytr': Ytrn, 'Xvl': Xvln, 'Yvl': Yvln, 'Xts': Xtsn, 'Yts': Ytsn}, scheduler='threads')
Xtr, Ytr = arrays['Xtr'], arrays['Ytr']
Xvl, Yvl = arrays['Xvl'], arrays['Yvl']
Xts, Yts = arrays['Xts'], arrays['Yts']
log.info("  Train %s  Val %s  Test %s", Xtr.shape, Xvl.shape, Xts.shape)

  Computing 6 arrays to NumPy (scheduler='threads')...
[########################################] | 100% Completed | 162.62 s


2026-03-10 16:26:28,946 | INFO |   Train (368, 12, 670, 480)  Val (46, 12, 670, 480)  Test (69, 12, 670, 480)


In [15]:
# Convert Numpy matrices to Groundwater Datasets

dstr = GroundwaterDataset(Xtr, Ytr, augment=False)
dsvl = GroundwaterDataset(Xvl, Yvl, augment=False)
dsts = GroundwaterDataset(Xts, Yts, augment=False)

loaderkw = dict(batch_size=CFG.BATCHSIZE, num_workers=CFG.NWORKERS, pin_memory=CFG.PINMEMORY)

# Pass Datasets into Dataloaders for Training, Validation and Testing

train_loader = DataLoader(dstr, shuffle=True, **loaderkw)
val_loader = DataLoader(dsvl, shuffle=False, **loaderkw)
test_loader = DataLoader(dsts, shuffle=False, **loaderkw)

  Dataset ready — 74,061 samples (T=368, H=670, W=480, patch=64, stride=32)
  Dataset ready — 6,119 samples (T=46, H=670, W=480, patch=64, stride=32)
  Dataset ready — 10,972 samples (T=69, H=670, W=480, patch=64, stride=32)


## Training Tests

In [16]:
from model import CNNLSTMForecaster
from train import train, validate
from forecast import seasonal_forecast, longterm_forecast, save_geotiff
from utils import load_checkpoint, compute_metrics, print_metrics, log

In [17]:
import torch
model = CNNLSTMForecaster(CFG).to(CFG.DEVICE)
model.eval()
x_batch, y_batch = next(iter(train_loader))  # one batch
x_batch = x_batch.to(CFG.DEVICE)
y_batch = y_batch.to(CFG.DEVICE)
with torch.no_grad():
    pred = model(x_batch)
print("x finite:", torch.isfinite(x_batch).all().item())
print("y finite:", torch.isfinite(y_batch).all().item())
print("pred finite:", torch.isfinite(pred).all().item())
print("x range:", float(x_batch.min()), float(x_batch.max()))
print("y range:", float(y_batch.min()), float(y_batch.max()))
print("pred range:", float(pred.min()), float(pred.max()))
print("pred shape:", pred.shape, "y shape:", y_batch.shape)

x finite: True
y finite: True
pred finite: True
x range: -3.1514034271240234 25.287073135375977
y range: -1.4384106397628784 2.8891499042510986
pred range: -3.5253686904907227 4.30056095123291
pred shape: torch.Size([8, 6, 1, 64, 64]) y shape: torch.Size([8, 6, 1, 64, 64])


In [18]:
print("X means:", Xmeans)
print("X stds:", Xstds)
print("min X std:", Xstds.min(), "max X std:", Xstds.max())

# After normalisation to NumPy:
print("Xtr range:", np.nanmin(Xtr), np.nanmax(Xtr))
print("Any non-finite in Xtr?", np.isfinite(Xtr).all())

X means: [ 2.1413528e+03  2.4681816e+01  8.1988120e-01  4.6528250e+06
  2.3937433e+01  2.5518198e+01 -2.5983450e+06 -1.3990354e+06
 -1.0390056e+00  1.7561959e+00  1.2731778e+01  1.1489434e+03]
X stds: [2.7520579e+03 3.8881111e+00 1.2170020e+00 1.0577464e+06 4.1572399e+00
 3.6768260e+00 6.8285438e+05 6.2509531e+05 2.7305433e-01 1.8304076e+00
 1.7387316e+01 2.4913801e+03]
min X std: 0.27305433 max X std: 1057746.4
Xtr range: -4.6832757 75.597115
Any non-finite in Xtr? False


In [19]:
model = CNNLSTMForecaster(CFG)
model.count_params()

  Trainable parameters: 14,904,102


14904102

In [20]:
torch.autograd.set_detect_anomaly(True)

In [21]:
history = train(model, train_loader, val_loader, CFG)

2026-03-10 16:26:56,190 | INFO | Using 2 GPUs via DataParallel
/media/ibtida/Data_02/Preprocessed_Data/Raster_Clip/train.py:84: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=False)
2026-03-10 16:26:57,552 | INFO | 
═════════════════════════════════════════════════════════════════
  Device: cuda  |  Epochs: 10  |  Patience: 15
  LR: 0.0001  |  Scheduler: cosine
═════════════════════════════════════════════════════════════════

/media/ibtida/Data_02/Preprocessed_Data/Raster_Clip/train.py:23: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=False):
2026-03-10 17:18:45,723 | INFO | Epoch   1/10 | Train 0.7206 | Val 4.4502 | NSE -4.5072 | RMSE 2.1097 | 3108.2s
2026-03-10 17:18:45,838 | INFO |   ✓ New best val loss: 4.45021 — checkpoint saved.
2026-03-10 18:08:17,501 | INFO | Epoch   2/

## Plotting

In [ ]:
import matplotlib.pyplot as plt

# Channel names in order of X[..., C, ...]
CHANNEL_NAMES = CFG.BIWEEKLYVARS + CFG.YEARLYVARS

def plot_time_step_channels(X, Y, timestamps, t_idx, title_prefix="Train"):
    """Plot all input channels and the target for a single time index.

    This lets you visually confirm that all variables for a given (year, doty)
    share the same spatial structures and are properly aligned.
    """
    year, doty = timestamps[t_idx]
    x = X[t_idx]            # (C, H, W)
    y = Y[t_idx, 0]         # (H, W)
    C = x.shape[0]

    ncols = 4
    nrows = int(np.ceil((C + 1) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
    axes = axes.ravel()

    # Plot each input channel
    for c in range(C):
        ax = axes[c]
        im = ax.imshow(x[c], cmap="viridis")
        name = CHANNEL_NAMES[c] if c < len(CHANNEL_NAMES) else f"ch{c}"
        ax.set_title(f"{title_prefix} X[{t_idx}] {name}\n{year}-doy{doty:03d}")
        ax.axis("off")
        fig.colorbar(im, ax=ax, shrink=0.6)

    # Plot target
    ax = axes[C]
    im = ax.imshow(y, cmap="RdBu_r")
    ax.set_title(f"{title_prefix} Y[{t_idx}] target\n{year}-doy{doty:03d}")
    ax.axis("off")
    fig.colorbar(im, ax=ax, shrink=0.6)

    # Hide any unused axes
    for k in range(C + 1, len(axes)):
        axes[k].axis("off")

    plt.tight_layout()
    plt.show()

def inspect_dataset_sample(dataset, timestamps, sample_idx=0, channel=0):
    """Inspect one dataset sample and its temporal alignment.

    - Prints the (year, doty) sequence for X and Y.
    - Plots one input channel through time and the target sequence.
    - Uses the original X/Y arrays with NaNs preserved so missing/out-of-domain
      regions are shown as transparent, not as 0.0.
    """
    # Recover the underlying time index for this sample
    t, i, j = dataset.indices[sample_idx]
    ps = dataset.ps
    seqlen = dataset.seqlen
    predhor = dataset.predhor

    print(f"Sample {sample_idx}: base t={t}, patch origin=(i={i}, j={j})")

    print("Input sequence timestamps (X):")
    for s in range(seqlen):
        year, doty = timestamps[t + s]
        print(f"  step {s:02d} -> year={year}, doy={doty:03d}")

    print("Target sequence timestamps (Y):")
    for s in range(predhor):
        year, doty = timestamps[t + seqlen + s]
        print(f"  step {s:02d} -> year={year}, doy={doty:03d}")

    # Work directly from underlying NumPy arrays (with NaNs), not from filled tensors
    X_full = dataset.X  # (T, C, H, W)
    Y_full = dataset.Y  # (T, 1, H, W)

    x_np = X_full[t : t + seqlen, :, i:i+ps, j:j+ps]
    y_np = Y_full[t + seqlen : t + seqlen + predhor, :, i:i+ps, j:j+ps]

    # Treat any configured NODATA value as missing too
    nodata = getattr(CFG, "NODATAVAL", None)
    if nodata is not None:
        x_np = np.where(x_np == nodata, np.nan, x_np)
        y_np = np.where(y_np == nodata, np.nan, y_np)

    # Plot
    fig, axes = plt.subplots(2, max(seqlen, predhor),
                             figsize=(3 * max(seqlen, predhor), 6))

    # First row: input sequence for one channel
    for s in range(seqlen):
        ax = axes[0, s]
        data = np.ma.masked_invalid(x_np[s, channel])  # mask NaNs
        im = ax.imshow(data, cmap="viridis")
        year, doty = timestamps[t + s]
        ax.set_title(f"X[{s}] ch{channel}\n{year}-doy{doty:03d}")
        ax.axis("off")
        fig.colorbar(im, ax=ax, shrink=0.6)

    # Second row: target sequence
    for s in range(predhor):
        ax = axes[1, s]
        data = np.ma.masked_invalid(y_np[s, 0])  # mask NaNs
        im = ax.imshow(data, cmap="RdBu_r")
        year, doty = timestamps[t + seqlen + s]
        ax.set_title(f"Y[{s}] target\n{year}-doy{doty:03d}")
        ax.axis("off")
        fig.colorbar(im, ax=ax, shrink=0.6)

    # Hide unused subplot slots
    for row in range(2):
        for col in range(max(seqlen, predhor)):
            if (row == 0 and col >= seqlen) or (row == 1 and col >= predhor):
                axes[row, col].axis("off")

    plt.tight_layout()
    plt.show()

# Usage examples (uncomment and run in the notebook):
#
# 1) Check per-(year, doty) spatial alignment of all channels + target
#    for a specific training timestep:
#
# t_idx = 0  # or any index < len(tr_ts)
# plot_time_step_channels(Xtr, Ytr, tr_ts, t_idx, title_prefix="Train")
#
# 2) Verify that a dataset sample uses a contiguous sequence of doty
#    values for X and the correct future doty values for Y:
#
# inspect_dataset_sample(dstr, tr_ts, sample_idx=0, channel=0)

In [ ]:
# Pick any time index in the training period
t_idx = 7      # or np.random.randint(0, len(tr_ts))

plot_time_step_channels(Xtr, Ytr, tr_ts, t_idx, title_prefix="Train")

In [ ]:
idx3 = 21

#plot_time_step_channels(Xvl, Yvl, vl_ts, idx2, title_prefix="Val")
plot_time_step_channels(Xvl, Yvl, vl_ts, idx3, title_prefix="Validation")

In [ ]:
idx2 = 12

#plot_time_step_channels(Xvl, Yvl, vl_ts, idx2, title_prefix="Val")
plot_time_step_channels(Xts, Yts, ts_ts, idx2, title_prefix="Test")

In [ ]:
s_idx = 12

inspect_dataset_sample(dstr, tr_ts, sample_idx=s_idx, channel=0)
inspect_dataset_sample(dstr, tr_ts, sample_idx=s_idx, channel=1)
inspect_dataset_sample(dstr, tr_ts, sample_idx=s_idx, channel=2)
inspect_dataset_sample(dstr, tr_ts, sample_idx=s_idx, channel=3)
inspect_dataset_sample(dstr, tr_ts, sample_idx=s_idx, channel=4)
inspect_dataset_sample(dstr, tr_ts, sample_idx=s_idx, channel=5)
inspect_dataset_sample(dstr, tr_ts, sample_idx=s_idx, channel=6)
inspect_dataset_sample(dstr, tr_ts, sample_idx=s_idx, channel=7)
inspect_dataset_sample(dstr, tr_ts, sample_idx=s_idx, channel=8)
inspect_dataset_sample(dstr, tr_ts, sample_idx=s_idx, channel=9)

## Dataloader Testing

In [ ]:
import torch

def show_batch(name, loader, max_batches=4):
    print(f"\n=== {name} ===")
    for b, (x, y) in enumerate(loader):
        # x: (B, SEQLEN, C, H_patch, W_patch)
        # y: (B, PRED_HORIZON, 1, H_patch, W_patch)
        print(f"Batch {b}:")
        print("  x.shape:", x.shape)
        print("  y.shape:", y.shape)
        print("  x dtype / device:", x.dtype, "/", x.device)
        print("  y dtype / device:", y.dtype, "/", y.device)

        # Simple stats on NDVI (channel 0) of first sample in batch
        ndvi = x[0, :, 0]  # (SEQLEN, H_patch, W_patch)
        print("  NDVI (ch0) over seq:")
        print("    min:", float(torch.min(ndvi)))
        print("    max:", float(torch.max(ndvi)))
        print("    mean:", float(torch.nanmean(ndvi)))

        if b + 1 >= max_batches:
            break

show_batch("Train loader", train_loader)
show_batch("Val loader",   val_loader)
show_batch("Test loader",  test_loader)

In [ ]:
import torch
import torch.nn as nn

class DummyModel(nn.Module):
    def __init__(self, in_channels, pred_horizon):
        super().__init__()
        self.pred_horizon = pred_horizon
        self.conv = nn.Conv2d(in_channels, 1, kernel_size=3, padding=1)

    def forward(self, x):
        # x: (B, T, C, H, W)
        B, T, C, H, W = x.shape
        x_flat = x.reshape(B * T, C, H, W)
        out = self.conv(x_flat).reshape(B, T, 1, H, W)  # (B, T, 1, H, W)

        # For this demo, pretend we forecast using the last `pred_horizon` steps
        # of the encoded sequence.
        if T < self.pred_horizon:
            raise ValueError(f"Sequence length {T} < pred_horizon {self.pred_horizon}")
        return out[:, -self.pred_horizon:]  # (B, pred_horizon, 1, H, W)

model = DummyModel(in_channels=CFG.NCHANNELS, pred_horizon=CFG.PREDHORIZON).to(CFG.DEVICE)
optim = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

model.train()
for batch_idx, (x, y) in enumerate(train_loader):
    x = x.to(CFG.DEVICE)          # (B, SEQLEN, C, H, W)
    y = y.to(CFG.DEVICE)          # (B, PRED_HORIZON, 1, H, W)

    pred = model(x)               # (B, PRED_HORIZON, 1, H, W)
    target = y                    # shapes now match exactly
    loss = loss_fn(pred, target)

    optim.zero_grad()
    loss.backward()
    optim.step()

    if batch_idx % 10 == 0:
        print(f"Batch {batch_idx:03d}  loss={loss.item():.4f}")

    if batch_idx >= 100:   # stop early; just a demo
        break